In [1]:
import os
from dotenv import load_dotenv
from reanimator.core import Reanimator
from reanimator.labelers import OpenAILabeler, LocalModelLabeler, TopicChunkPair, calculate_cohens_kappa
from reanimator.retrieval import Indexer, Retriever, reciprocal_rank_fusion, run_experiment
from reanimator.models import save_judgements, load_judgements, Document

from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
import pyterrier as pt
import nltk

load_dotenv()

import nltk
nltk.download('punkt_tab')

D:\User\Project\Reanimator\src\reanimator\extractors.py:184: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  if not caption or caption is "" or caption is None:
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\write\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [2]:
# Initialize Reanimator with dummy parameters (we won't use the download functionality)
reanimator = Reanimator(
    irds_name="irds:cord19/trec-covid",  # This won't be used
    email="dummy@gmail.com",
    config={
        "downloader": {
            "email": "dummy@gmail.com"
        }
    }
)

Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
D:\User\Project\Reanimator\src\reanimator\sources.py:18: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


INFO: OpenAILabeler initialized with model: gpt-4.1-mini-2025-04-14


In [3]:
# Skip the dataset loading and create a document for your ten.pdf
pdf_path = "data/pdfs/ten.pdf"  # Your PDF file path

# Create a document object manually
doc = Document(
    doc_id="ten_pdf",  # Custom document ID
    pdf_path=pdf_path  # Path to your PDF file
)

# Create a list with just this one document
docs = [doc]

# Set accelerator options
accelerator_options = AcceleratorOptions(
    num_threads=8, device=AcceleratorDevice.MPS
)

print(f"Using PDF file: {pdf_path}")

Using PDF file: data/pdfs/ten.pdf


In [4]:
# Extract content from the PDF (this will include formulas)
print(f"Extracting content from {pdf_path}...")
reanimator.extract_content(docs, accelerator_options)

# Check if formulas were extracted
if docs[0].formulas:
    print(f"\n✅ Success! Found {len(docs[0].formulas)} formulas:")
    print("=" * 60)
    
    for i, formula in enumerate(docs[0].formulas, 1):
        print(f"Formula {i}:")
        print(f"  Page: {formula.page}")
        print(f"  Text: {formula.text}")
        if formula.latex:
            print(f"  LaTeX: {formula.latex}")
        else:
            print(f"  LaTeX: Not available")
        print("-" * 40)
else:
    print("❌ No formulas found in the PDF.")
    print("This could mean:")
    print("1. The PDF doesn't contain mathematical formulas")
    print("2. The formulas are images rather than text")
    print("3. There might be an issue with the extraction")

# Save the extracted document
reanimator.save_documents(docs, "documents")
print(f"\nDocument saved to 'documents' folder")

Extracting content from data/pdfs/ten.pdf...

Step 3: Extracting content from PDFs...


Extracting Content:   0%|          | 0/1 [00:00<?, ?it/s]2025-09-22 23:06:52,071 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-09-22 23:06:54,209 - INFO - Going to convert document batch...
2025-09-22 23:06:54,211 - INFO - Initializing pipeline for StandardPdfPipeline with options hash b482b7b5f70e963efe771f8708610e1b
2025-09-22 23:06:54,234 - INFO - Loading plugin 'docling_defaults'
2025-09-22 23:06:54,238 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-09-22 23:06:54,258 - INFO - Loading plugin 'docling_defaults'
2025-09-22 23:06:54,268 - INFO - Registered ocr engines: ['easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2025-09-22 23:06:54,269 - WARNING - MPS is not available in the system. Fall back to 'CPU'
2025-09-22 23:06:54,269 - INFO - Accelerator device: 'cpu'
2025-09-22 23:06:56,102 - WARNING - MPS is not available in the system. Fall back to 'CPU'
2025-09-22 23:06:56,103 - INFO - Accelerator device: 'cpu'
2025-09-22 23:06:57,438 - WARNING -


✅ Success! Found 10 formulas:
Formula 1:
  Page: 3
  Text: \overset { \text {s of} } { \text {poly-} } \quad P ( H \, | \, D ) = \frac { P ( D \, | \, H ) P ( H ) } { P ( D ) } \ ,
  LaTeX: P( H | D ) = P( D | H )P( H ) P( D ) , (1)
----------------------------------------
Formula 2:
  Page: 3
  Text: \int _ { \real \cdot } d k \int _ { \real } ( \theta | D , H ) = \frac { P ( D | \theta , H ) P ( \theta | H ) } { P ( D | H ) } \ .
  LaTeX: P( θ | D,H ) = P( D | θ , H )P( θ | H ) P( D | H ) . (2)
----------------------------------------
Formula 3:
  Page: 3
  Text: \underset { \colon } { \text {and} } \quad \mathcal { E } \equiv P ( D \, | H ) = \int d ^ { n } \, \theta P ( D \, | \theta , H ) P ( \theta | H ) \ , \\ \text {seas-} \quad \underset { \colon } { \text {in} } \quad \text {where } \, \mathcal { E } \, \text { denotes the evidence of the hypothesis } \, H
  LaTeX: E ≡ P( D | H ) = ∫ d n θ P( D | θ , H )P( θ | H ) , (3)
----------------------------------------
Formula 4:
  P